In [ ]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import time

# 'data_files' folder එක ඇතුළේ ඇති බව සඳහන් කරන්න
df = pd.read_csv("data_files/data_files/diabetic_data.csv", na_values="?")

# නිවැරදිව load වී ඇත්දැයි බැලීමට (Null values ගණන පරීක්ෂා කරන්න)
# print(df.isnull().sum())

# get data
# df.head(10)
# df.info()

# check missing values percentage
df.isna().mean() * 100

# remove weight column
df.drop(columns=["weight"], inplace=True) 
print("Weight column removed.")

#  remove expired patients
expired_ids = [11, 19, 20]
df = df[~df["discharge_disposition_id"].isin(expired_ids)]
print(f"Remaining records: {len(df)}")

# remove duplicate rows
df = df.drop_duplicates()

# clean invalid gender values
df["gender"] = df["gender"].replace("Unknown/Invalid", np.nan)

# categorical columns correct 
categorical_cols = ["race", "gender", "age", "readmitted"]

for col in categorical_cols:
    df[col] = df[col].astype("category")



## phaase 2: Scrape data from website   


# වැඩිපුරම ඇති රෝග විනිශ්චය කේත 20 ලබා ගැනීම
top_20_codes = df['diag_1'].value_counts().head(20).index.tolist()
print(f"Scrape කිරීමට නියමිත කේත: {top_20_codes}")    


# වෙබ් අඩවියෙන් විස්තර scrape කිරීමේ function එක
def scrape_icd9_descriptions(codes):
    descriptions = {}
    base_url = "http://icd9.chrisendres.com/index.php?srchtype=procode&srchtext="
    
    for code in codes:
        try:
            # URL එක සකස් කර request එක යැවීම
            response = requests.get(f"{base_url}{code}")
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # වෙබ් අඩවියේ අදාළ විස්තරය ඇති කොටස සොයා ගැනීම
            # සටහන: වෙබ් අඩවියේ සැකැස්ම අනුව මෙය වෙනස් විය හැක. 
            # සාමාන්‍යයෙන් විස්තරය පවතින්නේ නිශ්චිත <div> හෝ <td> ටැගයකයි.
            desc_element = soup.find('div', {'id': 'content'}) # මෙය උදාහරණයක් පමණි
            
            if desc_element:
                descriptions[code] = desc_element.get_text(strip=True)[:100] # පළමු අකුරු 100
            else:
                descriptions[code] = "Description Not Found"
                
            print(f"කේතය {code} සඳහා විස්තරය ලබාගත්තා.")
            
            # තත්පර 1ක විවේකයක් ලබාදීම (Ethical Scraping) 
            time.sleep(1)
            
        except Exception as e:
            descriptions[code] = "Error retrieving description"
            print(f"Error for {code}: {e}")
            
    return descriptions

# ශ්‍රිතය ක්‍රියාත්මක කර විස්තර ලබා ගැනීම
scraped_data = scrape_icd9_descriptions(top_20_codes)







C:\Users\Shavindi\AppData\Local\Temp\ipykernel_22908\808837932.py:5: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data_files/data_files/diabetic_data.csv", na_values="?")


Weight column removed.
Remaining records: 100114
